## Currency Converter

In [1]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

import requests

In [2]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

# tool

@tool
def get_convesrion_factor(base_currency: str, target_currency: str) -> float:
    """
        This function fetches the currency conversion factor between the given base currency and the target currency.
    """

    url = f"https://v6.exchangerate-api.com/v6/4fbf9973bd4275857522a93b/pair/{base_currency}/{target_currency}"
    response = requests.get(url)

    return response.json()


@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
        Given a currency conversion rate. This function calculates the target currency value from a base currency value
    """

    return base_currency_value *  conversion_rate

In [3]:
### tool binding

llm = ChatGroq(model="openai/gpt-oss-120b")

In [4]:
llm_with_tools = llm.bind_tools([get_convesrion_factor, convert])

In [5]:
messages = [HumanMessage("What is the conversion factor between INR and USD, and based on that can you convert 10 usd to inr")]

In [6]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={})]

In [7]:
ai_message = llm_with_tools.invoke(messages)

In [8]:
ai_message

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need conversion factor between INR and USD. Use function get_convesrion_factor with base_currency "USD"? The user asks conversion factor between INR and USD. Likely base INR to USD or USD to INR? Then convert 10 USD to INR. So need factor USD->INR. Use get_convesrion_factor with base_currency "USD", target_currency "INR". Then use convert with base_currency_value 10? But convert function only takes base_currency_value, but doesn\'t accept which currencies? Probably it uses previously fetched factor? Might need to pass factor? The spec: "Given a currency conversion rate. This function calculates the target currency value from a base currency value". It expects a conversion rate set somewhere? Might need to pass base_currency_value and maybe it uses a stored rate? But we need to pass conversion factor? The function signature only has base_currency_value. No rate param. Maybe the function is preconfigured with the last fact

In [9]:
messages.append(ai_message)

In [10]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need conversion factor between INR and USD. Use function get_convesrion_factor with base_currency "USD"? The user asks conversion factor between INR and USD. Likely base INR to USD or USD to INR? Then convert 10 USD to INR. So need factor USD->INR. Use get_convesrion_factor with base_currency "USD", target_currency "INR". Then use convert with base_currency_value 10? But convert function only takes base_currency_value, but doesn\'t accept which currencies? Probably it uses previously fetched factor? Might need to pass factor? The spec: "Given a currency conversion rate. This function calculates the target currency value from a base currency value". It expects a conversion rate set somewhere? Might need to pass base_currency_value and maybe it uses a stor

In [11]:
ai_message.tool_calls

[{'name': 'get_convesrion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'fc_2b76b20c-8438-4805-8d79-5e5721d51f8f',
  'type': 'tool_call'}]

In [12]:

import json
from langchain_core.messages import ToolMessage

for tool_call in ai_message.tool_calls:

    if tool_call["name"] == "get_convesrion_factor":

        # Execute first tool
        tool_result = get_convesrion_factor.invoke(
            tool_call["args"]
        )

        # Extract conversion rate
        conversion_rate = tool_result["conversion_rate"]

        # Append proper ToolMessage
        messages.append(
            ToolMessage(
                content=json.dumps(tool_result),
                tool_call_id=tool_call["id"]
            )
        )

        # Execute second tool
        tool_result2 = convert.invoke({
            "base_currency_value": 10,
            "conversion_rate": conversion_rate
        })

        print("Conversion Result:", tool_result2)


Conversion Result: 940.857


In [18]:
from langchain_core.messages import ToolMessage

messages.append(response)

messages.append(
    ToolMessage(
        content=str(tool_result),
        tool_call_id=tool_call["id"]
    )
)

In [19]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need conversion factor between INR and USD. Use function get_convesrion_factor with base_currency "USD"? The user asks conversion factor between INR and USD. Likely base INR to USD or USD to INR? Then convert 10 USD to INR. So need factor USD->INR. Use get_convesrion_factor with base_currency "USD", target_currency "INR". Then use convert with base_currency_value 10? But convert function only takes base_currency_value, but doesn\'t accept which currencies? Probably it uses previously fetched factor? Might need to pass factor? The spec: "Given a currency conversion rate. This function calculates the target currency value from a base currency value". It expects a conversion rate set somewhere? Might need to pass base_currency_value and maybe it uses a stor

In [20]:
response = llm_with_tools.invoke(messages)
print(response.content)
print(response.content)
print(response.tool_calls)
print(response)

The current conversion factor from **USD → INR** is **1 USD = 94.0857 INR**.

So, converting **10 USD**:

\[
10 \text{ USD} \times 94.0857 = 940.857 \text{ INR}
\]

**≈ ₹ 940.86** (rounded to two decimal places).
The current conversion factor from **USD → INR** is **1 USD = 94.0857 INR**.

So, converting **10 USD**:

\[
10 \text{ USD} \times 94.0857 = 940.857 \text{ INR}
\]

**≈ ₹ 940.86** (rounded to two decimal places).
[]
content='The current conversion factor from **USD → INR** is **1\u202fUSD = 94.0857\u202fINR**.\n\nSo, converting **10\u202fUSD**:\n\n\\[\n10 \\text{ USD} \\times 94.0857 = 940.857 \\text{ INR}\n\\]\n\n**≈\u202f₹\u202f940.86** (rounded to two decimal places).' additional_kwargs={'reasoning_content': 'We have conversion factor USD to INR = 94.0857. Need to convert 10 USD to INR: 10 * 94.0857 = 940.857 INR. Provide answer.'} response_metadata={'token_usage': {'completion_tokens': 129, 'prompt_tokens': 547, 'total_tokens': 676, 'completion_time': 0.271016109, 'complet

In [21]:
response.content

'The current conversion factor from **USD → INR** is **1\u202fUSD = 94.0857\u202fINR**.\n\nSo, converting **10\u202fUSD**:\n\n\\[\n10 \\text{ USD} \\times 94.0857 = 940.857 \\text{ INR}\n\\]\n\n**≈\u202f₹\u202f940.86** (rounded to two decimal places).'